# <center>**Enhancing LEM-X Imaging with the IROS Reconstruction Pipeline**<center>

## <center>**Sky Reconstruction Efficiency**<center>

In [1]:
from pathlib import Path
from typing import Any, Callable

import numpy as np
import pandas as pd

from bloodmoon.mask import CodedMaskCamera, codedmask
from bloodmoon.io import simulation_files
from bloodmoon.types import CoordEquatorial
import darksun as ds
from darksun.data import Log, DataLoader, CatalogueLoader

from IROSrec.handle import config_dirpaths
import imgmaker as mgm
from imgmaker.fns import CameraUnitMap

In [2]:
# MASK_FITS: str = "mask_NTHT_20260129_CORRECTED.fits"
MASK_FITS: str = "mask_NTHT_20250725.fits"

# SKYFIELD: str = "IROSDummy"
SKYFIELD: str = "GalacticCentre"
# DATA_FITS: str = "baseline_2-50keV_1ks"
DATA_FITS: str = "galctr_rxte-sax_mask_050_1040x17_2-50keV_1ks_mask25"

RUN_ID: str = 'GC_IROS_doubleCam_rec_mask25'

ID_CAMERA_A: str = "cam1a"
ID_CAMERA_B: str = "cam1b"
DATASET: str = "reconstructed"

E_min: float = 2.0  # [keV]
E_max: float = 50.0  # [keV]
coords2exclude: list[CoordEquatorial] | None = None

UPS_X, UPS_Y = 2, 1
hide_bulk_els_y: float = 1.5   # [mm]

In [3]:
MASK_PATH, SIMUL_DATA_PATH, SAVE_PATH = config_dirpaths(
    mask=MASK_FITS,
    skyfield=SKYFIELD,
    simul=DATA_FITS,
    runID=RUN_ID,
)
OUT_RESULTS_PATH = mgm.config_savedata_to()

wfm: CodedMaskCamera = codedmask(MASK_PATH, UPS_X, UPS_Y, hide_bulk_els_y=hide_bulk_els_y)

filepaths: dict[str, dict[str, Path]] = simulation_files(SIMUL_DATA_PATH)
sdlA = ds.get_data(filepaths[ID_CAMERA_A][DATASET], E_min=E_min, E_max=E_max, coords=coords2exclude)
catA = ds.get_catalogue(filepaths[ID_CAMERA_A]['sources'])
sdlB = ds.get_data(filepaths[ID_CAMERA_B][DATASET], E_min=E_min, E_max=E_max, coords=coords2exclude)
catB = ds.get_catalogue(filepaths[ID_CAMERA_B]['sources'])

logA, logB = ds.load_database(f"{SAVE_PATH}/IROS_sources_db.fits")

# Loading data...
# Loading completed!


### <center>**Benchmark Tables**<center>

In [4]:
import re

def adjust_Tabfrmt(txt: str) -> str:
    # insert \hline instead of rules (journal guidelines)
    for rule in ('toprule', 'midrule', 'bottomrule'):
        txt = txt.replace(rule, 'hline')
    # shift caption and label at the end (journal guidelines)
    pattern = r"(\\begin\{table\}.*?)(\\caption\{.*?\})\s*(\\label\{.*?\})\s*(\\begin\{tabular\}.*?\\end\{tabular\})"
    replacement = r"\1\4\n\2\n\3"
    txt = re.sub(pattern, replacement, txt, flags=re.DOTALL)
    # convert to onecolumn
    txt = txt.replace('table', 'table*')
    return txt

def sort_by(df: pd.DataFrame, key: str, **kwargs: Any) -> pd.DataFrame:
    """Sort DataFrame wrt input column key."""
    return df.sort_values(by=[key], ascending=False, ignore_index=True, **kwargs)

In [5]:
from numpy.typing import NDArray

def comp_src_mstd(log: Log, varmap: NDArray, boxsize: tuple[int, int]) -> NDArray:
    """Computes the RMSE for each IROS source from given varmap in specified array box."""
    mstds: list[float] = []
    boxsize_ = (max(boxsize[0], 1), max(boxsize[1], 1))
    for y, x in zip(log.log['y'], log.log['x']):
        srows, scols = (
            slice(y - boxsize_[0], y + boxsize_[0] + 1),
            slice(x - boxsize_[1], x + boxsize_[1] + 1),
        )
        mstd = np.sqrt(np.mean(varmap[srows, scols]))
        mstds.append(mstd)
    return np.array(mstds)


def gather_cam_data(
    log: Log,
    catalogue: CatalogueLoader,
    sdl: DataLoader,
    camera: CodedMaskCamera,
    varmap: NDArray,
) -> pd.DataFrame:
    """
    Gathers single camera data from IROS reconstruction database.
    """
    ids = np.array([src.upper() for src in log.log['ID']])
    theta_res_x, theta_res_y = mgm.get_angularcoords_residues(log, catalogue, sdl, camera)
    cts = np.array(log.log['fluence']).round(decimals=0)
    true_cts = mgm.extract_catalogue_fluences(log, catalogue, sdl, camera)
    src_mstd = comp_src_mstd(log, varmap, boxsize=tuple(int(np.ceil(a // 2)) for a in ds.psf_extension(camera)[::-1]))
    dmap = {
        'Source': ids,
        'DthetaX': theta_res_x,
        'DthetaY': theta_res_y,
        'True_cts': true_cts,
        'IROS_cts': cts,
        'Dcts': (cts - true_cts) / src_mstd,
        'SNR': np.array(log.log['snr']),
    }
    return pd.DataFrame(dmap)


def get_unit_tab(data_camA: pd.DataFrame, data_camB: pd.DataFrame) -> pd.DataFrame:
    """Generates a Dataframe with output data from analysed LEM-X Unit."""
    # NOTE: data relative to not associated sources is DROPPED
    df = pd.merge(data_camA.dropna(), data_camB.dropna(), on='Source', how='outer', suffixes=('_A', '_B'))
    snrA, snrB = map(lambda col: df[col] ** 2, ('SNR_A', 'SNR_B'))
    df['SNR'] = np.sqrt(snrA.add(snrB, fill_value=0.0))
    df = df.drop(columns=['SNR_A', 'SNR_B'])
    return sort_by(df, 'SNR')

In [6]:
from bloodmoon.mask import count, variance

def get_varmap(camera: CodedMaskCamera, sdl: DataLoader) -> NDArray:
    detector = count(camera, sdl.DLdata)[0]
    varmap = variance(camera, detector)
    return varmap


varmapA, varmapB = map(lambda sdl: get_varmap(wfm, sdl), (sdlA, sdlB))

UserInfo: using bulk mask of [0.0 x 1.5] mm.


In [7]:
ds.pixels_angular_resolution(wfm)
cu_map = mgm.get_srcmap_for_unit(logA.log['ID'], logB.log['ID'])

# Table - CAMERA A
data_camA = gather_cam_data(logA, catA, sdlA, wfm, varmapA)

# Table - CAMERA B
data_camB = gather_cam_data(logB, catB, sdlB, wfm, varmapB)


Pixel angular resolution at upscaling (x, y): (2, 1)
  - fine direction: 2.1163 arcmin
  - coarse direction: 8.4653 arcmin



Analysing LEMX-CAM1BS6: 100%|██████████| 24/24 [00:05<00:00,  4.70it/s]  


In [8]:
unit_data = get_unit_tab(data_camA, data_camB)

KWS = {
    'label': 'Table1',
    'caption': 'Testing $`to\\_latex`$ fn.',
    'float_format': "%.4f",
    'column_format': 'l' + 'c' * (len(unit_data.columns) - 2) + 'r',
}
tab = mgm.df2TeXtab(
    df=sort_by(unit_data, 'SNR'),
    adjust_tabfrmt=adjust_Tabfrmt,
    # save_to=f'{OUT_RESULTS_PATH}/../texTable_Unit_results_{DATASET}_{E_min}-{E_max}keV_mask25.tex',
    overwrite=True,
    **KWS,
)

In [9]:
unit_data

,Source,DthetaX_A,DthetaY_A,True_cts_A,IROS_cts_A,Dcts_A,DthetaX_B,DthetaY_B,True_cts_B,IROS_cts_B,Dcts_B,SNR
0,SCOX1,0.134392,0.000842,1053180.0,1029329.0,-18.417862,1.167945,6.624501,941404.0,687480.0,-206.049799,857.027554
1,GX5-1,0.128884,0.613478,114662.0,107560.0,-5.042647,0.044085,3.412037,116083.0,113588.0,-1.808130,105.030486
2,GX17+2,0.313859,3.234636,72138.0,72944.0,0.577558,-0.922349,1.352233,77028.0,76300.0,-0.528392,63.422135
3,GX349+2,-0.110878,-0.916299,83098.0,80207.0,-2.051783,0.282521,0.738575,82486.0,66846.0,-11.331453,62.023007
4,GX9+1,0.081148,1.151556,63099.0,64859.0,1.249050,0.228215,-2.690714,64219.0,74759.0,7.637943,61.938511
5,GX340+0,-0.191633,10.218775,45694.0,43044.0,-1.982790,0.390893,5.594752,48752.0,41895.0,-5.126680,40.738818
6,X1820-303,-0.336037,8.433669,34826.0,38179.0,2.379753,-0.454206,16.599523,33914.0,38989.0,3.676977,33.410574
7,GX13+1,0.181342,10.043429,36663.0,36765.0,0.072389,-0.078528,14.666229,36745.0,30345.0,-4.636821,32.102993
8,GX3+1,0.417556,-6.702535,35923.0,36431.0,0.360607,-0.103794,-5.562135,37099.0,38585.0,1.076756,31.568836
9,CIRX1,-0.477577,1.857133,20979.0,20128.0,-0.948228,0.971086,3.343676,21991.0,21686.0,-0.358411,29.384246


In [10]:
unit_data.dropna()

,Source,DthetaX_A,DthetaY_A,True_cts_A,IROS_cts_A,Dcts_A,DthetaX_B,DthetaY_B,True_cts_B,IROS_cts_B,Dcts_B,SNR
0,SCOX1,0.134392,0.000842,1053180.0,1029329.0,-18.417862,1.167945,6.624501,941404.0,687480.0,-206.049799,857.027554
1,GX5-1,0.128884,0.613478,114662.0,107560.0,-5.042647,0.044085,3.412037,116083.0,113588.0,-1.808130,105.030486
2,GX17+2,0.313859,3.234636,72138.0,72944.0,0.577558,-0.922349,1.352233,77028.0,76300.0,-0.528392,63.422135
3,GX349+2,-0.110878,-0.916299,83098.0,80207.0,-2.051783,0.282521,0.738575,82486.0,66846.0,-11.331453,62.023007
4,GX9+1,0.081148,1.151556,63099.0,64859.0,1.249050,0.228215,-2.690714,64219.0,74759.0,7.637943,61.938511
5,GX340+0,-0.191633,10.218775,45694.0,43044.0,-1.982790,0.390893,5.594752,48752.0,41895.0,-5.126680,40.738818
6,X1820-303,-0.336037,8.433669,34826.0,38179.0,2.379753,-0.454206,16.599523,33914.0,38989.0,3.676977,33.410574
7,GX13+1,0.181342,10.043429,36663.0,36765.0,0.072389,-0.078528,14.666229,36745.0,30345.0,-4.636821,32.102993
8,GX3+1,0.417556,-6.702535,35923.0,36431.0,0.360607,-0.103794,-5.562135,37099.0,38585.0,1.076756,31.568836
9,CIRX1,-0.477577,1.857133,20979.0,20128.0,-0.948228,0.971086,3.343676,21991.0,21686.0,-0.358411,29.384246
